# Практика: AE, conditional AE и VAE на KMNIST

В этой работе мы последовательно построим три модели:

1. **AE (Autoencoder)**
2. **cAE (conditional Autoencoder)**
3. **VAE (Variational Autoencoder)**

Все модели обучаются на **KMNIST** — наборе из 10 классов рукописных японских символов размером $28\times28$.

## 1. Импорты и воспроизводимость


In [ ]:
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm


SEED = 42


def seed_everything(seed: int = SEED) -> None:
    """Фиксирует основные источники случайности."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Устройство:", device)

## 2. Данные: KMNIST

`ToTensor()` выполняет две операции:

- переводит изображение в тензор формы `[channels, height, width]`;
- масштабирует значения пикселей из диапазона `0…255` в `0…1`.

Это важно, потому что на выходе декодера будет `Sigmoid`, также возвращающий значения от 0 до 1.


In [ ]:
@dataclass
class Config:
    latent_dim: int = 16
    num_classes: int = 10
    batch_size: int = 128
    learning_rate: float = 1e-3
    epochs: int = 5
    beta: float = 1.0
    fast_mode: bool = False


cfg = Config()

transform = transforms.ToTensor()

# При первом запуске torchvision скачает датасет в папку ./data.
full_train_dataset = datasets.KMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.KMNIST(
    root="./data", train=False, download=True, transform=transform
)

# Сначала делаем честное фиксированное train/validation-разбиение.
split_generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [55_000, 5_000],
    generator=split_generator,
)

if cfg.fast_mode:
    # Индексы берём из зафиксированной случайной перестановки, а не первые подряд.
    subset_generator = torch.Generator().manual_seed(SEED)
    train_indices = torch.randperm(len(train_dataset), generator=subset_generator)[:12_000]
    val_indices = torch.randperm(len(val_dataset), generator=subset_generator)[:2_000]
    test_indices = torch.randperm(len(test_dataset), generator=subset_generator)[:2_000]
    train_dataset = Subset(train_dataset, train_indices.tolist())
    val_dataset = Subset(val_dataset, val_indices.tolist())
    test_dataset = Subset(test_dataset, test_indices.tolist())

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0
)

print(f"Train: {len(train_dataset):,}")
print(f"Validation: {len(val_dataset):,}")
print(f"Test: {len(test_dataset):,}")

In [ ]:
def show_batch(loader: DataLoader, n: int = 10) -> None:
    """Показывает первые n изображений одного батча."""
    images, labels = next(iter(loader))
    n = min(n, len(images))

    fig, axes = plt.subplots(1, n, figsize=(1.5 * n, 2))
    for ax, image, label in zip(axes, images[:n], labels[:n]):
        ax.imshow(image.squeeze(0), cmap="gray")
        ax.set_title(f"class {label.item()}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_batch(train_loader)

## 3. Общие сверточные блоки

Все три модели используют одинаковую основу.

### Энкодер

Две свёртки со `stride=2` последовательно уменьшают размер изображения:

```text
[B, 1, 28, 28] → [B, 32, 14, 14] → [B, 64, 7, 7]
```

### Декодер

`ConvTranspose2d` выполняет обучаемое увеличение пространственного размера:

```text
[B, 64, 7, 7] → [B, 32, 14, 14] → [B, 1, 28, 28]
```

`LeakyReLU` сохраняет небольшой градиент для отрицательных значений. Финальный `Sigmoid` ограничивает реконструкцию диапазоном `[0, 1]`.

In [ ]:
class ConvEncoder(nn.Module):
    """Общая сверточная часть энкодера для изображений 1×28×28."""

    feature_dim = 64 * 7 * 7

    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ConvDecoder(nn.Module):
    """Общая сверточная часть декодера; принимает плоский вектор признаков."""

    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.project = nn.Linear(input_dim, 64 * 7 * 7)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = self.project(z)
        x = x.view(-1, 64, 7, 7)
        return self.net(x)

## 4. Обычный автоэнкодер (AE)

AE решает задачу реконструкции:

$$
z = E_\phi(x), \qquad \hat{x}=D_\theta(z).
$$

Энкодер превращает изображение в компактный вектор `z`, а декодер восстанавливает изображение. Мы минимизируем бинарную кросс-энтропию между каждым исходным пикселем и его реконструкцией.

Латентные коды обычного AE **не обязаны** образовывать непрерывное пространство или подчиняться известному распределению. Поэтому случайный вектор из нормального распределения, переданный декодеру AE, обычно не даёт осмысленное изображение.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim: int = 16) -> None:
        super().__init__()
        self.encoder_features = ConvEncoder()
        self.to_latent = nn.Linear(ConvEncoder.feature_dim, latent_dim)
        self.decoder = ConvDecoder(latent_dim)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Возвращает детерминированный латентный код изображения."""
        features = self.encoder_features(x)
        return self.to_latent(features)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Восстанавливает изображение по латентному коду."""
        return self.decoder(z)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Полный путь image → latent code → reconstruction."""
        z = self.encode(x)
        reconstruction = self.decode(z)
        return reconstruction, z

## 5. Условный автоэнкодер (cAE)

В cAE декодер получает не только код `z`, но и условие `y` — номер класса. Метка переводится в one-hot-вектор и конкатенируется с латентным кодом:

$$
\hat{x}=D_\theta([z;\operatorname{onehot}(y)]).
$$

Условие отвечает на вопрос «объект какого класса нужно восстановить?». Во время обучения подаётся правильная метка. После обучения можно зафиксировать `z` и менять только `y`, исследуя, насколько модель научилась использовать условие.

Распределение `z` никак не регуляризуется, поэтому произвольная случайная точка может лежать далеко от кодов обучающих объектов.

In [ ]:
class ConditionalAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 16, num_classes: int = 10) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.encoder_features = ConvEncoder()
        self.to_latent = nn.Linear(ConvEncoder.feature_dim, latent_dim)

        # Декодер получает latent_dim чисел кода и num_classes чисел one-hot-метки.
        self.decoder = ConvDecoder(latent_dim + num_classes)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        features = self.encoder_features(x)
        return self.to_latent(features)

    def decode(self, z: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        # one_hot сначала имеет целочисленный тип; приводим его к типу z.
        condition = F.one_hot(labels, num_classes=self.num_classes).to(z.dtype)
        conditioned_z = torch.cat([z, condition], dim=1)
        return self.decoder(conditioned_z)

    def forward(
        self, x: torch.Tensor, labels: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        z = self.encode(x)
        reconstruction = self.decode(z, labels)
        return reconstruction, z

## 6. Вариационный автоэнкодер (VAE)

VAE кодирует изображение параметрами нормального распределения:

$$
q_\phi(z\mid x)=\mathcal{N}\left(\mu(x),\operatorname{diag}(\sigma^2(x))\right).
$$

Из этого распределения нужно получить случайный `z`, сохранив возможность обратного распространения ошибки. Для этого используется **трюк репараметризации**:

$$
\varepsilon\sim\mathcal{N}(0,I),\qquad
z=\mu+\sigma\odot\varepsilon,
\qquad \sigma=\exp(\tfrac12\log\sigma^2).
$$

Функция потерь содержит две части:

$$
\mathcal{L}=\mathcal{L}_{reconstruction}+\beta\,D_{KL}
\left(q_\phi(z\mid x)\;\|\;\mathcal{N}(0,I)\right).
$$

- reconstruction loss требует сохранять информацию об исходном изображении;
- KL-дивергенция приближает латентное распределение к стандартному нормальному;
- `beta` управляет компромиссом. Слишком малое значение ухудшает структуру пространства, слишком большое может ухудшить реконструкцию.

In [ ]:
class VariationalAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 16) -> None:
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder_features = ConvEncoder()

        # Два выхода описывают среднее и логарифм дисперсии q(z|x).
        self.to_mu = nn.Linear(ConvEncoder.feature_dim, latent_dim)
        self.to_log_var = nn.Linear(ConvEncoder.feature_dim, latent_dim)
        self.decoder = ConvDecoder(latent_dim)

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        features = self.encoder_features(x)
        return self.to_mu(features), self.to_log_var(features)

    def reparameterize(
        self, mu: torch.Tensor, log_var: torch.Tensor
    ) -> torch.Tensor:
        """Дифференцируемое семплирование z из N(mu, sigma²)."""
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    def forward(
        self, x: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        reconstruction = self.decode(z)
        return reconstruction, mu, log_var

## 7. Функции потерь

Для всех моделей используем `binary_cross_entropy` с `reduction="sum"`, а затем делим сумму на размер батча. Поэтому loss интерпретируется как средняя сумма ошибки по 784 пикселям одного объекта.

Для VAE KL-дивергенция между диагональным нормальным распределением энкодера и $\mathcal N(0,I)$ имеет аналитическую форму — её не нужно оценивать семплированием.

In [ ]:
def reconstruction_loss(
    reconstruction: torch.Tensor, images: torch.Tensor
) -> torch.Tensor:
    """Средняя по батчу сумма BCE по всем пикселям изображения."""
    return F.binary_cross_entropy(reconstruction, images, reduction="sum") / images.size(0)


def vae_loss(
    reconstruction: torch.Tensor,
    images: torch.Tensor,
    mu: torch.Tensor,
    log_var: torch.Tensor,
    beta: float = 1.0,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Возвращает общий loss и обе его части для раздельного анализа."""
    recon = reconstruction_loss(reconstruction, images)

    # KL(q(z|x) || N(0, I)); суммируем по latent_dim и усредняем по batch.
    kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    kl = kl / images.size(0)

    total = recon + beta * kl
    return total, recon, kl

## 8. Универсальный цикл обучения

1. перенос батча на вычислительное устройство;
2. обнуление старых градиентов;
3. прямой проход;
4. вычисление loss;
5. `backward()` — расчёт градиентов;
6. `optimizer.step()` — обновление параметров.


In [ ]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    model_kind: str,
    optimizer: torch.optim.Optimizer | None = None,
    beta: float = 1.0,
) -> dict[str, float]:
    """Выполняет одну обучающую или валидационную эпоху."""
    is_training = optimizer is not None
    model.train(is_training)

    totals = {"loss": 0.0, "reconstruction": 0.0, "kl": 0.0}
    object_count = 0

    context = torch.enable_grad() if is_training else torch.no_grad()
    with context:
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            batch_size = images.size(0)

            if is_training:
                optimizer.zero_grad(set_to_none=True)

            if model_kind == "ae":
                reconstruction, _ = model(images)
                loss = reconstruction_loss(reconstruction, images)
                recon_value = loss.detach()
                kl_value = torch.zeros((), device=device)
            elif model_kind == "cae":
                reconstruction, _ = model(images, labels)
                loss = reconstruction_loss(reconstruction, images)
                recon_value = loss.detach()
                kl_value = torch.zeros((), device=device)
            elif model_kind == "vae":
                reconstruction, mu, log_var = model(images)
                loss, recon_value, kl_value = vae_loss(
                    reconstruction, images, mu, log_var, beta=beta
                )
            else:
                raise ValueError(f"Неизвестный model_kind: {model_kind}")

            if is_training:
                loss.backward()
                optimizer.step()

            # loss уже усреднён по батчу; возвращаем взвешенное среднее по объектам.
            totals["loss"] += loss.item() * batch_size
            totals["reconstruction"] += recon_value.item() * batch_size
            totals["kl"] += kl_value.item() * batch_size
            object_count += batch_size

    return {key: value / object_count for key, value in totals.items()}


def fit_model(
    model: nn.Module,
    model_kind: str,
    epochs: int,
    learning_rate: float,
    beta: float = 1.0,
) -> tuple[nn.Module, pd.DataFrame]:
    """Обучает модель и восстанавливает параметры лучшей validation-эпохи."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = []
    best_val_loss = float("inf")
    best_state = None

    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(
            model, train_loader, model_kind, optimizer=optimizer, beta=beta
        )
        val_metrics = run_epoch(
            model, val_loader, model_kind, optimizer=None, beta=beta
        )

        row = {
            "epoch": epoch,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }

        print(
            f"Epoch {epoch:02d}/{epochs}: "
            f"train={train_metrics['loss']:.3f}, "
            f"val={val_metrics['loss']:.3f}"
        )

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


def plot_history(history: pd.DataFrame, title: str) -> None:
    plt.figure(figsize=(7, 4))
    plt.plot(history["epoch"], history["train_loss"], marker="o", label="train")
    plt.plot(history["epoch"], history["val_loss"], marker="o", label="validation")
    plt.xlabel("Эпоха")
    plt.ylabel("Loss на объект")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

## 9. Обучение AE

При одинаковом `latent_dim` AE обычно показывает хорошую реконструкцию, потому что вся функция потерь направлена только на восстановление. Однако это ещё не означает, что пространство между кодами заполнено осмысленно.

In [ ]:
seed_everything()
ae = Autoencoder(latent_dim=cfg.latent_dim)
ae, ae_history = fit_model(
    ae,
    model_kind="ae",
    epochs=cfg.epochs,
    learning_rate=cfg.learning_rate,
)
plot_history(ae_history, "AE: обучение")

## 10. Обучение cAE

Число классов равно 10, поэтому условие добавляет к латентному вектору 10 компонент. Для корректного сравнения остальные параметры обучения совпадают с AE.

In [ ]:
seed_everything()
cae = ConditionalAutoencoder(
    latent_dim=cfg.latent_dim, num_classes=cfg.num_classes
)
cae, cae_history = fit_model(
    cae,
    model_kind="cae",
    epochs=cfg.epochs,
    learning_rate=cfg.learning_rate,
)
plot_history(cae_history, "Conditional AE: обучение")

## 11. Обучение VAE


In [ ]:
seed_everything()
vae = VariationalAutoencoder(latent_dim=cfg.latent_dim)
vae, vae_history = fit_model(
    vae,
    model_kind="vae",
    epochs=cfg.epochs,
    learning_rate=cfg.learning_rate,
    beta=cfg.beta,
)
plot_history(vae_history, f"VAE: обучение, beta={cfg.beta}")

# Для VAE полезно посмотреть составляющие loss отдельно.
vae_history[["epoch", "train_reconstruction", "train_kl", "val_reconstruction", "val_kl"]]

## 12. Визуальное сравнение реконструкций

Во время оценки VAE используем среднее `mu`, а не случайный `z`. Это убирает случайный шум и делает сравнение реконструкций воспроизводимым.

In [ ]:
@torch.no_grad()
def reconstruct_all(
    ae: Autoencoder,
    cae: ConditionalAutoencoder,
    vae: VariationalAutoencoder,
    loader: DataLoader,
    n: int = 10,
) -> None:
    ae.eval()
    cae.eval()
    vae.eval()

    images, labels = next(iter(loader))
    images = images[:n].to(device)
    labels = labels[:n].to(device)

    ae_recon, _ = ae(images)
    cae_recon, _ = cae(images, labels)
    mu, _ = vae.encode(images)
    vae_recon = vae.decode(mu)

    rows = [images, ae_recon, cae_recon, vae_recon]
    row_names = ["Original", "AE", "cAE", "VAE"]

    fig, axes = plt.subplots(4, n, figsize=(1.35 * n, 5.8))
    for row_idx, (batch, name) in enumerate(zip(rows, row_names)):
        for col_idx in range(n):
            axes[row_idx, col_idx].imshow(
                batch[col_idx].squeeze().cpu(), cmap="gray", vmin=0, vmax=1
            )
            axes[row_idx, col_idx].axis("off")
            if row_idx == 0:
                axes[row_idx, col_idx].set_title(str(labels[col_idx].item()))
        axes[row_idx, 0].set_ylabel(name, rotation=0, labelpad=35, va="center")

    plt.suptitle("Исходные изображения и реконструкции; сверху указан класс")
    plt.tight_layout()
    plt.show()


reconstruct_all(ae, cae, vae, test_loader)

## 13. Численное сравнение

Для всех моделей вычисляем одинаковую величину — среднюю MSE на один пиксель.

Низкая MSE не делает модель автоматически лучшей: VAE может немного уступать AE в реконструкции, но выигрывать в возможности семплирования и структуре латентного пространства.

In [ ]:
@torch.no_grad()
def evaluate_mse(model: nn.Module, loader: DataLoader, model_kind: str) -> float:
    model.eval()
    squared_error = 0.0
    pixel_count = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if model_kind == "ae":
            reconstruction, _ = model(images)
        elif model_kind == "cae":
            reconstruction, _ = model(images, labels)
        elif model_kind == "vae":
            mu, _ = model.encode(images)
            reconstruction = model.decode(mu)
        else:
            raise ValueError(model_kind)

        squared_error += F.mse_loss(
            reconstruction, images, reduction="sum"
        ).item()
        pixel_count += images.numel()

    return squared_error / pixel_count


comparison = pd.DataFrame(
    [
        {"model": "AE", "test_mse_per_pixel": evaluate_mse(ae, test_loader, "ae")},
        {"model": "cAE", "test_mse_per_pixel": evaluate_mse(cae, test_loader, "cae")},
        {"model": "VAE", "test_mse_per_pixel": evaluate_mse(vae, test_loader, "vae")},
    ]
).sort_values("test_mse_per_pixel")

comparison

## 14. Проверка влияния условия в cAE

Берём одно изображение, кодируем его один раз и декодируем тот же `z` со всеми десятью метками. Если условие используется моделью, изменение метки должно систематически менять форму результата.

Не следует ожидать идеальных представителей каждого класса: cAE обучался реконструировать изображение с его правильной меткой и не получал отдельного штрафа за качество «контрфактических» комбинаций `(z, другая метка)`.

In [ ]:
@torch.no_grad()
def visualize_condition_effect(
    model: ConditionalAutoencoder, loader: DataLoader
) -> None:
    model.eval()
    images, labels = next(iter(loader))
    image = images[:1].to(device)
    true_label = labels[0].item()

    z = model.encode(image)
    repeated_z = z.repeat(cfg.num_classes, 1)
    all_labels = torch.arange(cfg.num_classes, device=device)
    conditioned_outputs = model.decode(repeated_z, all_labels)

    fig, axes = plt.subplots(1, cfg.num_classes + 1, figsize=(16, 2))
    axes[0].imshow(image[0, 0].cpu(), cmap="gray")
    axes[0].set_title(f"input\nclass {true_label}")
    axes[0].axis("off")

    for class_id in range(cfg.num_classes):
        axes[class_id + 1].imshow(
            conditioned_outputs[class_id, 0].cpu(), cmap="gray", vmin=0, vmax=1
        )
        axes[class_id + 1].set_title(f"y={class_id}")
        axes[class_id + 1].axis("off")

    plt.suptitle("Один латентный код + разные условия")
    plt.tight_layout()
    plt.show()


visualize_condition_effect(cae, test_loader)

## 15. Генерация новых изображений с VAE

KL-составляющая приближает распределение латентных кодов к $\mathcal N(0,I)$. Поэтому после обучения можно семплировать `z` из стандартного нормального распределения и передавать его декодеру без исходного изображения.

Это принципиальное отличие VAE от обычного AE и cAE.

In [ ]:
@torch.no_grad()
def sample_vae(model: VariationalAutoencoder, n: int = 20) -> None:
    model.eval()
    z = torch.randn(n, model.latent_dim, device=device)
    samples = model.decode(z).cpu()

    columns = 10
    rows = int(np.ceil(n / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(12, 1.4 * rows))
    axes = np.asarray(axes).reshape(-1)

    for index, ax in enumerate(axes):
        ax.axis("off")
        if index < n:
            ax.imshow(samples[index, 0], cmap="gray", vmin=0, vmax=1)

    plt.suptitle("Случайные изображения: z ~ N(0, I)")
    plt.tight_layout()
    plt.show()


sample_vae(vae)

## 16. Интерполяция в латентном пространстве

Линейная интерполяция строит точки между кодами двух реальных изображений:

$$
z(t)=(1-t)z_1+t z_2, \qquad t\in[0,1].
$$

Плавное изменение декодированных изображений — качественный признак того, что локальная область латентного пространства организована осмысленно. Для VAE используем средние `mu` двух апостериорных распределений.

In [ ]:
@torch.no_grad()
def interpolate_vae(
    model: VariationalAutoencoder, loader: DataLoader, steps: int = 12
) -> None:
    model.eval()
    images, labels = next(iter(loader))

    # Выбираем два изображения разных классов.
    first_index = 0
    second_index = next(
        index for index, label in enumerate(labels) if label != labels[first_index]
    )
    pair = images[[first_index, second_index]].to(device)
    pair_labels = labels[[first_index, second_index]].tolist()

    mu, _ = model.encode(pair)
    t = torch.linspace(0, 1, steps, device=device).unsqueeze(1)
    interpolated_z = (1 - t) * mu[0] + t * mu[1]
    outputs = model.decode(interpolated_z).cpu()

    fig, axes = plt.subplots(1, steps, figsize=(1.35 * steps, 2))
    for index, ax in enumerate(axes):
        ax.imshow(outputs[index, 0], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{index / (steps - 1):.1f}")
        ax.axis("off")
    plt.suptitle(f"VAE-интерполяция: class {pair_labels[0]} → class {pair_labels[1]}")
    plt.tight_layout()
    plt.show()


interpolate_vae(vae, test_loader)

## 17. Двумерная проекция латентного пространства

У моделей `latent_dim=16`, поэтому для изображения пространства используем PCA. PCA находит два направления с наибольшей дисперсией.

Для VAE в качестве представления объекта берём `mu`, поскольку это центр распределения `q(z|x)` и он не содержит случайности отдельного семпла.

In [ ]:
@torch.no_grad()
def collect_latents(
    model: nn.Module,
    loader: DataLoader,
    model_kind: str,
    max_objects: int = 2_000,
) -> tuple[torch.Tensor, torch.Tensor]:
    model.eval()
    latent_batches = []
    label_batches = []
    collected = 0

    for images, labels in loader:
        images = images.to(device)
        if model_kind in {"ae", "cae"}:
            z = model.encode(images)
        elif model_kind == "vae":
            z, _ = model.encode(images)  # z здесь означает mu
        else:
            raise ValueError(model_kind)

        latent_batches.append(z.cpu())
        label_batches.append(labels)
        collected += len(images)
        if collected >= max_objects:
            break

    return (
        torch.cat(latent_batches)[:max_objects],
        torch.cat(label_batches)[:max_objects],
    )


def pca_2d(latents: torch.Tensor) -> torch.Tensor:
    """Центрирует данные и проецирует их на первые две главные компоненты."""
    centered = latents - latents.mean(dim=0, keepdim=True)
    # V содержит главные направления пространства признаков.
    _, _, V = torch.pca_lowrank(centered, q=2)
    return centered @ V[:, :2]


def plot_latent_spaces() -> None:
    models = [(ae, "ae", "AE"), (cae, "cae", "cAE"), (vae, "vae", "VAE")]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    for ax, (model, kind, title) in zip(axes, models):
        latents, labels = collect_latents(model, test_loader, kind)
        points = pca_2d(latents)
        scatter = ax.scatter(
            points[:, 0], points[:, 1], c=labels, cmap="tab10", s=8, alpha=0.65
        )
        ax.set_title(title)
        ax.set_xlabel("PC 1")
        ax.set_ylabel("PC 2")
        ax.grid(alpha=0.2)

    handles, _ = scatter.legend_elements(num=10)
    fig.legend(handles, [str(i) for i in range(10)], title="Класс", loc="center right")
    plt.suptitle("PCA-проекция латентных представлений")
    plt.tight_layout(rect=[0, 0, 0.94, 1])
    plt.show()


plot_latent_spaces()

## 18. Расширенный эксперимент с гиперпараметрами

В базовом запуске мы обучили по одной модели каждого типа.

- AE с `latent_dim = 2, 8, 32` показывает влияние пропускной способности bottleneck;
- VAE с `beta = 0.1, 1, 5` показывает компромисс между реконструкцией и регулярностью пространства.


In [ ]:
RUN_EXTENDED_EXPERIMENTS = True


def run_extended_experiments() -> tuple[pd.DataFrame, pd.DataFrame]:
    experiment_epochs = 3 if cfg.fast_mode else 10
    ae_rows = []
    vae_rows = []

    for latent_dim in [2, 8, 32]:
        seed_everything()
        experiment_model = Autoencoder(latent_dim=latent_dim)
        experiment_model, history = fit_model(
            experiment_model,
            model_kind="ae",
            epochs=experiment_epochs,
            learning_rate=cfg.learning_rate,
        )
        ae_rows.append(
            {
                "latent_dim": latent_dim,
                "best_val_loss": history["val_loss"].min(),
                "test_mse_per_pixel": evaluate_mse(
                    experiment_model, test_loader, "ae"
                ),
            }
        )

    for beta in [0.1, 1.0, 5.0]:
        seed_everything()
        experiment_model = VariationalAutoencoder(latent_dim=cfg.latent_dim)
        experiment_model, history = fit_model(
            experiment_model,
            model_kind="vae",
            epochs=experiment_epochs,
            learning_rate=cfg.learning_rate,
            beta=beta,
        )
        best_row = history.loc[history["val_loss"].idxmin()]
        vae_rows.append(
            {
                "beta": beta,
                "val_reconstruction": best_row["val_reconstruction"],
                "val_kl": best_row["val_kl"],
                "test_mse_per_pixel": evaluate_mse(
                    experiment_model, test_loader, "vae"
                ),
            }
        )

    return pd.DataFrame(ae_rows), pd.DataFrame(vae_rows)


if RUN_EXTENDED_EXPERIMENTS:
    ae_experiment, vae_experiment = run_extended_experiments()
    display(ae_experiment)
    display(vae_experiment)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(
        ae_experiment["latent_dim"],
        ae_experiment["test_mse_per_pixel"],
        marker="o",
    )
    axes[0].set_title("AE: влияние latent_dim")
    axes[0].set_xlabel("latent_dim")
    axes[0].set_ylabel("test MSE на пиксель")
    axes[0].grid(alpha=0.3)

    axes[1].plot(
        vae_experiment["beta"],
        vae_experiment["test_mse_per_pixel"],
        marker="o",
        label="reconstruction MSE",
    )
    axes[1].set_xscale("log")
    axes[1].set_title("VAE: влияние beta")
    axes[1].set_xlabel("beta (логарифмическая шкала)")
    axes[1].set_ylabel("test MSE на пиксель")
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 19. Сохранение моделей


In [ ]:
torch.save(ae.state_dict(), "ae_kmnist.pt")
torch.save(cae.state_dict(), "cae_kmnist.pt")
torch.save(vae.state_dict(), "vae_kmnist.pt")

print("Сохранены ae_kmnist.pt, cae_kmnist.pt и vae_kmnist.pt")